# **Gold Layer Data Aggregation**

In [1]:
silver_df = spark.read.table("silver_matched_fonts")

StatementMeta(, 444395f8-707d-4121-91d5-e6fd25f225a7, 3, Finished, Available, Finished, False)

#### **Build the summary table — match confidence breakdown**

In [2]:
from pyspark.sql import functions as F

total_records = silver_df.count()

confidence_summary = silver_df.groupBy("match_type").agg(
    F.count("*").alias("record_count")
).withColumn(
    "percentage", F.round((F.col("record_count") / total_records) * 100,1)
)

display(confidence_summary)
confidence_summary.write.format("delta").mode("overwrite").saveAsTable("gold_confidence_summary")

StatementMeta(, 444395f8-707d-4121-91d5-e6fd25f225a7, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 074d5a13-dc64-45d0-8b9a-bc8efb40a545)

#### **Foundry coverage — how many distinct foundries got matched**

In [3]:
foundry_coverage = silver_df.filter(F.col("matched_foundry").isNotNull()) \
    .groupBy("matched_foundry") \
    .agg(F.count("*").alias("matched_fonts")) \
    .orderBy(F.desc("matched_fonts"))

display(foundry_coverage)
foundry_coverage.write.format("delta").mode("overwrite").saveAsTable("gold_foundry_coverage")


StatementMeta(, 444395f8-707d-4121-91d5-e6fd25f225a7, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8073cf8-029a-431a-9f9d-851bdab10309)

#### **Top unresolved records — the "needs review" list**

In [4]:
unresolved_records = silver_df.filter(F.col("match_type") == "unresolved") \
    .select("raw_font_name", "cleaned_name", "confidence_score") \
    .orderBy(F.desc("confidence_score"))

display(unresolved_records)
unresolved_records.write.format("delta").mode("overwrite").saveAsTable("gold_unresolved_records")


StatementMeta(, 444395f8-707d-4121-91d5-e6fd25f225a7, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c3fac467-19a1-499d-99a4-16397b0012c2)

#### **Average confidence by style type**

In [5]:
avg_confidence_by_style = silver_df.filter(F.col("confidence_score").isNotNull()) \
    .groupBy("style") \
    .agg(F.round(F.avg("confidence_score"), 1).alias("avg_confidence"))

display(avg_confidence_by_style)
avg_confidence_by_style.write.format("delta").mode("overwrite").saveAsTable("gold_avg_confidence_by_style")


StatementMeta(, 444395f8-707d-4121-91d5-e6fd25f225a7, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b0f212db-5716-449a-ab69-a080a0996e5f)